# GitHub incidents activity calendar

Filter GitHub incidents and explore daily incident counts or total duration in a calendar view.

In [ ]:
from pathlib import Path
import re

import pandas as pd
import mercury as mr

CSV_URL = "https://raw.githubusercontent.com/mrshu/github-statuses/refs/heads/master/parsed/downtime_windows.csv"


def extract_components(title: str) -> str:
    if not isinstance(title, str) or not title.strip():
        return ""

    patterns = [
        r"^Incident with (.+)$",
        r"^Scheduled Maintenance for (.+)$",
        r"^(.+?) maintenance$",
        r"^(.+?) (?:is|are) Paused$",
    ]

    text = None
    for pattern in patterns:
        match = re.match(pattern, title.strip(), flags=re.IGNORECASE)
        if match:
            text = match.group(1)
            break

    if text is None:
        return ""

    text = re.sub(r"\band\b", ",", text, flags=re.IGNORECASE)
    parts = [p.strip(" .") for p in text.split(",") if p.strip(" .")]

    seen = set()
    cleaned = []
    for p in parts:
        if p not in seen:
            seen.add(p)
            cleaned.append(p)

    return "|".join(cleaned)


incidents = pd.read_csv(CSV_URL)
incidents["start"] = pd.to_datetime(incidents["downtime_start"], utc=True)
incidents["end"] = pd.to_datetime(incidents["downtime_end"], utc=True)
incidents["date"] = incidents["start"].dt.tz_convert(None).dt.normalize()
incidents["duration_hours"] = incidents["duration_minutes"].fillna(0) / 60
incidents["components"] = incidents["title"].apply(extract_components)


In [ ]:
first_date = incidents["date"].min().date().isoformat()
last_date = incidents["date"].max().date().isoformat()
impact_choices = ["All"] + sorted(incidents["impact"].dropna().unique().tolist())

component_values = sorted(
    {
        item.strip()
        for value in incidents["components"].fillna("")
        for item in value.split("|")
        if item.strip()
    }
)
component_choices = ["All"] + component_values


In [ ]:
date_range = mr.DateRange(
    label="Incident date range",
    value=[first_date, last_date],
    min=first_date,
    max=last_date,
    start_url_key="from",
    end_url_key="to",
)

In [ ]:
impact_filter = mr.Select(
    label="Impact",
    value="All",
    choices=impact_choices,
    url_key="impact",
)

In [ ]:
component_filter = mr.Select(
    label="Component",
    value="All",
    choices=component_choices,
    url_key="component",
)

In [ ]:
metric_filter = mr.Select(
    label="Calendar metric",
    value="Incident count",
    choices=["Incident count", "Duration (hours)"],
    url_key="metric",
)

In [ ]:
color_filter = mr.Select(
    label="Calendar color",
    value="Green",
    choices=["Green", "Red"],
    url_key="color",
)

In [ ]:
selected_start = pd.Timestamp(date_range.value[0] or first_date)
selected_end = pd.Timestamp(date_range.value[1] or last_date)

filtered = incidents[
    incidents["date"].between(selected_start, selected_end)
].copy()

if impact_filter.value != "All":
    filtered = filtered[filtered["impact"] == impact_filter.value]

if component_filter.value != "All":
    selected_component = component_filter.value
    filtered = filtered[
        filtered["components"].fillna("").str.split("|", regex=False).apply(
            lambda values: selected_component in values
        )
    ]

if metric_filter.value == "Incident count":
    daily = filtered.groupby("date").size().rename("value").reset_index()
    calendar_title = "GitHub incidents starting each day"
    calendar_unit = "incidents"
else:
    daily = (
        filtered.groupby("date", as_index=False)["duration_hours"]
        .sum()
        .rename(columns={"duration_hours": "value"})
    )
    calendar_title = "Duration of GitHub incidents starting each day"
    calendar_unit = "hours"

if daily.empty:
    daily = pd.DataFrame({"date": [selected_start], "value": [0.0]})

In [ ]:
mr.Indicator([
    mr.Indicator(len(filtered), label="Filtered incidents"),
    mr.Indicator(
        f"{filtered['duration_hours'].sum():,.1f}",
        label="Total duration (hours)",
    ),
    mr.Indicator(
        filtered["date"].nunique(),
        label="Days with incidents",
    ),
])

In [ ]:
mr.ActivityCalendar(
    daily,
    date="date",
    value="value",
    title=calendar_title,
    unit=calendar_unit,
    color=color_filter.value.lower(),
    start_date=selected_start,
    end_date=selected_end,
)

## Filtered incidents

Duration is assigned to the day on which an incident started.

In [ ]:
incident_table = filtered.sort_values("start", ascending=False)[
    ["title", "start", "duration_minutes", "impact"]
].copy()
incident_table["start"] = incident_table["start"].dt.strftime("%Y-%m-%d %H:%M UTC")

incident_table.head(50)